# Part 4: Training pipeline

Baseline RoBERTa for PCL binary classification. Dev-only evaluation; save checkpoints, predictions, and metrics.

In [6]:
import ast
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score
from torch.utils.data import Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer

DATA_DIR = Path("../data")
CHECKPOINT_DIR = Path("../checkpoints")
OUTPUT_DIR = Path("../outputs")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cpu


## Load and prepare data

In [7]:
pcl = pd.read_csv(
    DATA_DIR / "dontpatronizeme_pcl.tsv",
    sep="\t",
    header=None,
    names=["id", "par_id", "keyword", "country", "text", "category"],
)
pcl["id"] = pcl["id"].astype(int)

def to_binary(label_str):
    vec = ast.literal_eval(label_str)
    return 1 if any(x == 1 for x in vec) else 0

train_labels = pd.read_csv(DATA_DIR / "train_semeval_parids-labels.csv")
train_labels["binary_label"] = train_labels["label"].apply(to_binary)

dev_labels = pd.read_csv(DATA_DIR / "dev_semeval_parids-labels.csv")
dev_labels["binary_label"] = dev_labels["label"].apply(to_binary)

train_df = pcl[pcl["id"].isin(train_labels["par_id"])].copy()
train_df = train_df.merge(
    train_labels[["par_id", "binary_label"]],
    left_on="id",
    right_on="par_id",
    how="inner",
)
train_df = train_df[["id", "text", "binary_label"]]

dev_df = pcl[pcl["id"].isin(dev_labels["par_id"])].copy()
dev_df = dev_df.merge(
    dev_labels[["par_id", "binary_label"]],
    left_on="id",
    right_on="par_id",
    how="inner",
)
dev_df = dev_df[["id", "text", "binary_label"]]

def valid_text(x):
    return isinstance(x, str) and len(x.strip()) > 0

train_df = train_df[train_df["text"].apply(valid_text)].reset_index(drop=True)
dev_df = dev_df[dev_df["text"].apply(valid_text)].reset_index(drop=True)

train_texts = train_df["text"].tolist()
train_labels_list = train_df["binary_label"].values
dev_texts = dev_df["text"].tolist()
dev_labels_list = dev_df["binary_label"].values

print(f"Train: {len(train_texts)} examples")
print(f"Dev: {len(dev_texts)} examples")

Train: 8375 examples
Dev: 2093 examples


## Tokenize and build dataloaders

In [8]:
MODEL_NAME = "roberta-base"
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_enc = tokenizer(
    train_texts,
    padding="max_length",
    truncation=True,
    max_length=MAX_LENGTH,
    return_tensors=None,
)
dev_enc = tokenizer(
    dev_texts,
    padding="max_length",
    truncation=True,
    max_length=MAX_LENGTH,
    return_tensors=None,
)

In [9]:
class PCLDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {
            k: torch.tensor(v[idx], dtype=torch.long)
            for k, v in self.encodings.items()
        }
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_dataset = PCLDataset(train_enc, train_labels_list)
dev_dataset = PCLDataset(dev_enc, dev_labels_list)

BATCH_SIZE = 16
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)
dev_loader = torch.utils.data.DataLoader(
    dev_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

## Baseline: one epoch (no class weights)

In [10]:
def evaluate(model, loader):
    model.eval()
    all_preds = []
    all_probs = []
    all_labels = []
    with torch.no_grad():
        for batch in loader:
            inputs = {
                k: v.to(DEVICE) for k, v in batch.items()
                if k != "labels"
            }
            labels = batch["labels"].to(DEVICE)
            out = model(**inputs)
            logits = out.logits
            probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            preds = logits.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_probs.extend(probs)
            all_labels.extend(labels.cpu().numpy())
    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)
    f1 = f1_score(all_labels, all_preds, pos_label=1)
    return f1, all_preds, all_probs, all_labels

In [11]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
criterion = torch.nn.CrossEntropyLoss()

run_name = "baseline_1epoch"
n_epochs = 1

for epoch in range(n_epochs):
    model.train()
    for batch in train_loader:
        inputs = {k: v.to(DEVICE) for k, v in batch.items() if k != "labels"}
        labels = batch["labels"].to(DEVICE)
        optimizer.zero_grad()
        out = model(**inputs)
        loss = criterion(out.logits, labels)
        loss.backward()
        optimizer.step()

dev_f1, dev_preds, dev_probs, dev_gold = evaluate(model, dev_loader)
print(f"Dev F1 (positive class): {dev_f1:.4f}")

run_dir = CHECKPOINT_DIR / run_name
run_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(run_dir)
tokenizer.save_pretrained(run_dir)

pd.DataFrame({
    "id": dev_df["id"].values,
    "gold": dev_gold,
    "pred": dev_preds,
    "prob_pos": dev_probs,
}).to_csv(OUTPUT_DIR / f"dev_{run_name}.csv", index=False)

metrics = {"run": run_name, "dev_f1": float(dev_f1)}
with open(OUTPUT_DIR / f"metrics_{run_name}.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Saved checkpoint to {run_dir}")
print(f"Saved dev predictions to {OUTPUT_DIR / ('dev_' + run_name + '.csv')}")

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 2519.06it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consid

Dev F1 (positive class): 0.0000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.89it/s]


Saved checkpoint to ../checkpoints/baseline_1epoch
Saved dev predictions to ../outputs/dev_baseline_1epoch.csv


## Run with class weights

In [12]:
counts = np.bincount(train_labels_list)
n_total = len(train_labels_list)
class_weights = torch.tensor(
    [n_total / (2 * counts[0]), n_total / (2 * counts[1])],
    dtype=torch.float32,
).to(DEVICE)
print(f"Class weights: {class_weights.cpu().numpy()}")

model_cw = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model_cw.to(DEVICE)
optimizer_cw = torch.optim.AdamW(model_cw.parameters(), lr=2e-5)
criterion_cw = torch.nn.CrossEntropyLoss(weight=class_weights)

run_name_cw = "class_weights_3epoch"
n_epochs_cw = 3
best_f1 = 0.0

for epoch in range(n_epochs_cw):
    model_cw.train()
    for batch in train_loader:
        inputs = {k: v.to(DEVICE) for k, v in batch.items() if k != "labels"}
        labels = batch["labels"].to(DEVICE)
        optimizer_cw.zero_grad()
        out = model_cw(**inputs)
        loss = criterion_cw(out.logits, labels)
        loss.backward()
        optimizer_cw.step()

    dev_f1_cw, _, _, _ = evaluate(model_cw, dev_loader)
    print(f"Epoch {epoch + 1} — Dev F1: {dev_f1_cw:.4f}")
    if dev_f1_cw > best_f1:
        best_f1 = dev_f1_cw
        run_dir_cw = CHECKPOINT_DIR / run_name_cw
        run_dir_cw.mkdir(parents=True, exist_ok=True)
        model_cw.save_pretrained(run_dir_cw)
        tokenizer.save_pretrained(run_dir_cw)

run_dir_cw = CHECKPOINT_DIR / run_name_cw
model_cw = AutoModelForSequenceClassification.from_pretrained(run_dir_cw)
model_cw.to(DEVICE)
dev_f1_cw, dev_preds_cw, dev_probs_cw, dev_gold_cw = evaluate(model_cw, dev_loader)

print(f"Best Dev F1: {best_f1:.4f}")

pd.DataFrame({
    "id": dev_df["id"].values,
    "gold": dev_gold_cw,
    "pred": dev_preds_cw,
    "prob_pos": dev_probs_cw,
}).to_csv(OUTPUT_DIR / f"dev_{run_name_cw}.csv", index=False)

metrics_cw = {"run": run_name_cw, "dev_f1": float(best_f1)}
with open(OUTPUT_DIR / f"metrics_{run_name_cw}.json", "w") as f:
    json.dump(metrics_cw, f, indent=2)
print(f"Saved best checkpoint to {CHECKPOINT_DIR / run_name_cw}")

Class weights: [0.55236775 5.2739296 ]


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1580.46it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consid

Epoch 1 — Dev F1: 0.4411


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.77it/s]


Epoch 2 — Dev F1: 0.5022


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  6.74it/s]


Epoch 3 — Dev F1: 0.4647


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2549.86it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              


Best Dev F1: 0.5022
Saved best checkpoint to ../checkpoints/class_weights_3epoch


## Next: Threshold tuning and dev.txt

Best model is class-weights (epoch 2, F1 0.5022). Sweep decision thresholds on dev probabilities to maximise F1, then write `dev.txt`.

In [13]:
df = pd.read_csv(OUTPUT_DIR / "dev_class_weights_3epoch.csv")
gold = df["gold"].values
prob_pos = df["prob_pos"].values

best_thresh = 0.5
best_f1 = 0.0
for thresh in [0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6]:
    pred = (prob_pos >= thresh).astype(int)
    f1 = f1_score(gold, pred, pos_label=1)
    print(f"Threshold {thresh:.2f} — Dev F1: {f1:.4f}")
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = thresh

print(f"\nBest threshold: {best_thresh} (Dev F1: {best_f1:.4f})")

Threshold 0.20 — Dev F1: 0.3928
Threshold 0.25 — Dev F1: 0.4099
Threshold 0.30 — Dev F1: 0.4274
Threshold 0.35 — Dev F1: 0.4422
Threshold 0.40 — Dev F1: 0.4575
Threshold 0.45 — Dev F1: 0.4825
Threshold 0.50 — Dev F1: 0.5022
Threshold 0.55 — Dev F1: 0.5147
Threshold 0.60 — Dev F1: 0.5322

Best threshold: 0.6 (Dev F1: 0.5322)


In [14]:
pred_best = (prob_pos >= best_thresh).astype(int)
DEV_TXT = Path("..") / "dev.txt"
with open(DEV_TXT, "w") as f:
    for p in pred_best:
        f.write(f"{p}\n")
print(f"Wrote {len(pred_best)} predictions to {DEV_TXT}")

Wrote 2093 predictions to ../dev.txt


## Test set predictions → test.txt

Load test data, run best model (class_weights checkpoint), apply same threshold, write `test.txt`. Run threshold-tuning and dev.txt cells first so `best_thresh` is set.

In [15]:
run_dir_cw = CHECKPOINT_DIR / "class_weights_3epoch"
tokenizer_test = AutoTokenizer.from_pretrained(run_dir_cw)

test_raw = pd.read_csv(
    DATA_DIR / "task4_test.tsv",
    sep="\t",
    header=None,
    names=["test_id", "par_id", "keyword", "country", "text"],
)
test_df = test_raw[test_raw["text"].apply(valid_text)].reset_index(drop=True)
test_texts = test_df["text"].tolist()
print(f"Test: {len(test_texts)} examples")

test_enc = tokenizer_test(
    test_texts,
    padding="max_length",
    truncation=True,
    max_length=MAX_LENGTH,
    return_tensors=None,
)
test_dataset = PCLDataset(test_enc, np.zeros(len(test_texts), dtype=np.int64))
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

model_test = AutoModelForSequenceClassification.from_pretrained(run_dir_cw)
model_test.to(DEVICE)
model_test.eval()

all_probs_test = []
with torch.no_grad():
    for batch in test_loader:
        inputs = {k: v.to(DEVICE) for k, v in batch.items() if k != "labels"}
        out = model_test(**inputs)
        probs = torch.softmax(out.logits, dim=1)[:, 1].cpu().numpy()
        all_probs_test.extend(probs)
all_probs_test = np.array(all_probs_test)

thresh = best_thresh
test_preds = (all_probs_test >= thresh).astype(int)
TEST_TXT = Path("..") / "test.txt"
with open(TEST_TXT, "w") as f:
    for p in test_preds:
        f.write(f"{p}\n")
print(f"Wrote {len(test_preds)} predictions to {TEST_TXT} (threshold={thresh})")

Test: 3832 examples


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2340.88it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              


Wrote 3832 predictions to ../test.txt (threshold=0.6)
